# MNIST CNN Pure Inference on PYNQ-Z2
Run this notebook **on the PYNQ-Z2 board** (Jupyter at 192.168.2.99)

Requirements:
- `cnn.bit` + `cnn.hwh` in the same directory as this notebook
- `fpga_weights/` folder (w_conv1.npy, b_conv1.npy, etc.) in the same directory
- An image file named `test_image.png` (e.g. a hand-drawn digit)


In [ ]:
import numpy as np
import time
from PIL import Image
import matplotlib.pyplot as plt
from pynq import Overlay, allocate
import pynq.lib.dma


In [ ]:
# Load the FPGA overlay (bitstream + hardware handoff)
print('Loading overlay...')
ol = Overlay('./cnn.bit')
print('Overlay loaded.')
dma = ol.axi_dma_0
cnn = ol.cnn_top_0


In [ ]:
# Load quantised weights (int16) and allocate contiguous DDR buffers
SCALE = 1024
weight_names = ['w_conv1', 'b_conv1', 'w_conv2', 'b_conv2', 'w_fc', 'b_fc']
bufs = {}

for name in weight_names:
    arr = np.load(f'fpga_weights/{name}.npy')
    buf = allocate(shape=arr.shape, dtype=np.int16)
    buf[:] = arr
    bufs[name] = buf

REG = {
    'w_conv1': 0x10, 'b_conv1': 0x1c,
    'w_conv2': 0x28, 'b_conv2': 0x34,
    'w_fc': 0x40,    'b_fc': 0x4c,
}
for name in weight_names:
    cnn.write(REG[name], bufs[name].physical_address & 0xFFFFFFFF)

print('Weights configured in FPGA registers.')


In [ ]:
def preprocess(img_uint8):
    x = img_uint8.astype(np.float32) / 255.0
    x = (x - 0.1307) / 0.3081
    return np.clip(np.round(x * SCALE), -32768, 32767).astype(np.int16)

in_buf  = allocate(shape=(784,), dtype=np.int16)
out_buf = allocate(shape=(10,),  dtype=np.int16)

def run_fpga_inference(img_array):
    in_buf[:] = preprocess(img_array.reshape(784))
    cnn.write(0x00, 1)
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    return out_buf[:].astype(np.float32) / SCALE


In [ ]:
# Load your own test image (e.g. upload 'test_image.png' to Jupyter)
try:
    img = Image.open('test_image.png').convert('L').resize((28, 28))
    img_array = np.array(img)
    
    # Run Hardware Inference
    logits = run_fpga_inference(img_array)
    pred = np.argmax(logits)
    
    # Display Results
    plt.imshow(img_array, cmap='gray')
    plt.title(f'Hardware Prediction: {pred}')
    plt.axis('off'); plt.show()
    print(f'Raw Logits: {np.round(logits, 2)}')
except FileNotFoundError:
    print('Please upload a test_image.png file to the same directory.')


In [ ]:
for name in weight_names:
    bufs[name].freebuffer()
in_buf.freebuffer()
out_buf.freebuffer()
